# 02 - MobileNet & MLP Training (Arabic)
Train MobileNetV2 on images and a lightweight MLP on MediaPipe keypoints.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, applications, models, callbacks
import matplotlib.pyplot as plt


## Configuration


In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')
DATASET_DIR = '/kaggle/input/arabic-sign-language-letters' if IS_KAGGLE else './dataset'
CSV_PATH = 'arabic_mediapipe_keypoints.csv'

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

MOBILENET_SAVE_PATH = 'mobilenet_arabic_final.h5'
MLP_SAVE_PATH = 'arsl_mediapipe_mlp_model_final.h5'
CLASSES_JSON = 'arabic_classes.json'


## Section A: MobileNet Training
### Data Loading & Splitting


In [ ]:
if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f"DATASET_DIR '{DATASET_DIR}' not found!")

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
with open(CLASSES_JSON, 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=4, ensure_ascii=False)
print(f"Saved {len(class_names)} classes to {CLASSES_JSON}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


### MobileNet Model Construction


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

base_model = applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
mobilenet_m = models.Model(inputs, outputs)

mobilenet_m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])


### Phase 1: Train Head


In [ ]:
history_phase1 = mobilenet_m.fit(train_ds, validation_data=val_ds, epochs=10)


### Phase 2: Fine-tuning


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

mobilenet_m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_phase2 = mobilenet_m.fit(train_ds, validation_data=val_ds, epochs=10,
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

mobilenet_m.save(MOBILENET_SAVE_PATH)
print(f"Saved MobileNet to {MOBILENET_SAVE_PATH}")

pd.DataFrame(history_phase2.history).to_csv('training_history_mobilenet.csv', index=False)


## Section B: MLP Training on Keypoints


In [ ]:
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"{CSV_PATH} missing. Run Notebook 01 first.")

df = pd.read_csv(CSV_PATH, encoding='utf-8')
print(f"Loaded {len(df)} rows from CSV.")

with open(CLASSES_JSON, 'r', encoding='utf-8') as f:
    mlp_classes = json.load(f)

# Ensure ML labels map correctly
missing_classes = set(df['label'].unique()) - set(mlp_classes)
if missing_classes:
    raise ValueError(f"Classes in CSV not found in {CLASSES_JSON}: {missing_classes}")

label_map = {name: i for i, name in enumerate(mlp_classes)}
df['label_idx'] = df['label'].map(label_map)

X = df.drop(['label', 'label_idx'], axis=1).values
y = df['label_idx'].values

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)


### MLP Model


In [ ]:
mlp_model = models.Sequential([
    layers.InputLayer(input_shape=(63,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(mlp_classes), activation='softmax')
])

mlp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_mlp = mlp_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50,
                            callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

mlp_model.save(MLP_SAVE_PATH)
pd.DataFrame(history_mlp.history).to_csv('training_history_mlp.csv', index=False)
print(f"Saved MLP to {MLP_SAVE_PATH}")
